# 01 — Evaluation Foundations

**No API key needed.** Pure Python + the standard library.

---

## The one idea behind every agent evaluation

Stripped of all the jargon, evaluating an agent is just this:

```
score = compare(what_the_agent_said, what_you_expected)
```

You run that comparison over many examples, average the scores, and now you
have a **number** that tells you how good your agent is. Change a prompt, re-run,
and watch the number move. That's the whole game.

This notebook builds the comparison functions — the **metrics** — by hand, so
that when you later reach for a library like `ragas` or `deepeval`, you know
exactly what's happening under the hood.

### What you'll build
1. A tiny **dataset** of test cases
2. Five **metrics**: exact match, normalized match, keyword coverage, token-F1, latency & cost
3. A reusable **eval harness** that ties them together into a report

## Step 1 — A test case is just input + expected output

Before you can score anything, you need *ground truth*: for a given input, what
is a good answer? We store each case as a small dictionary. Real teams keep
hundreds of these in a file — this is the seed of a **golden dataset** (which we
build out fully in notebook 05).

In [ ]:
# Each test case: what we send in, and what a good answer looks like.
dataset = [
    {"id": 1, "input": "What is the capital of France?",      "expected": "Paris"},
    {"id": 2, "input": "What is 12 multiplied by 12?",         "expected": "144"},
    {"id": 3, "input": "Who wrote the play Romeo and Juliet?", "expected": "William Shakespeare"},
    {"id": 4, "input": "What is the largest planet?",          "expected": "Jupiter"},
]

print(f"Loaded {len(dataset)} test cases.")
for case in dataset:
    print(f"  [{case['id']}] {case['input']!r} -> expects {case['expected']!r}")

## Step 2 — A 'system under test' to grade

We need something to evaluate. To keep this notebook LLM-free, our 'agent' is a
plain lookup function. It deliberately gets one answer *slightly* wrong and one
*completely* wrong — so our metrics have something interesting to catch.

> In a real project this function would call your actual agent. The eval code
> around it doesn't care what's inside — that's the point.

In [ ]:
def my_agent(question: str) -> str:
    """A fake 'agent'. In real life this would call your LLM / chain / graph."""
    answers = {
        "What is the capital of France?":      "The capital of France is Paris.",  # correct, but wordy
        "What is 12 multiplied by 12?":        "144",                              # exactly right
        "Who wrote the play Romeo and Juliet?": "Shakespeare",                      # right person, partial name
        "What is the largest planet?":          "Saturn",                          # just wrong
    }
    return answers.get(question, "I don't know.")

# Sanity check
for case in dataset:
    print(f"Q: {case['input']}")
    print(f"A: {my_agent(case['input'])}\n")

## Step 3 — Metric #1: Exact Match

The strictest metric: is the output *character-for-character* identical to the
expected answer? It's harsh (`"Paris"` != `"The capital of France is Paris."`)
but it's the right tool when there's exactly one acceptable string — an ID, a
yes/no, a classification label.

In [ ]:
def exact_match(prediction: str, expected: str) -> float:
    """1.0 if the strings are identical, else 0.0."""
    return 1.0 if prediction == expected else 0.0

print(exact_match("144", "144"))                  # 1.0  ✅
print(exact_match("Shakespeare", "William Shakespeare"))  # 0.0  ❌ too strict

## Step 4 — Metric #2: Normalized Match

Most 'wrong' answers in Step 3 aren't really wrong — they just differ in case,
punctuation, or surrounding whitespace. **Normalizing** both strings before
comparing fixes the easy mismatches: lowercase everything, strip punctuation,
and collapse spaces.

In [ ]:
import re

def normalize(text: str) -> str:
    text = text.lower().strip()
    text = re.sub(r"[^a-z0-9 ]", "", text)   # drop punctuation
    text = re.sub(r"\s+", " ", text)         # collapse whitespace
    return text

def normalized_match(prediction: str, expected: str) -> float:
    return 1.0 if normalize(prediction) == normalize(expected) else 0.0

print(normalized_match("Paris.", "paris"))            # 1.0  ✅ now matches
print(normalized_match("The capital is Paris.", "Paris"))  # 0.0  still too strict — extra words

## Step 5 — Metric #3: Keyword Coverage (`contains`)

For wordy answers, what we usually care about is: **did the key fact appear
anywhere in the response?** 'The capital of France is Paris.' clearly contains
the right answer. This metric checks whether the expected string is a substring
of the (normalized) prediction.

In [ ]:
def contains_match(prediction: str, expected: str) -> float:
    """1.0 if the expected answer appears anywhere in the prediction."""
    return 1.0 if normalize(expected) in normalize(prediction) else 0.0

print(contains_match("The capital of France is Paris.", "Paris"))  # 1.0  ✅
print(contains_match("Shakespeare", "William Shakespeare"))        # 0.0  expected is longer than prediction
print(contains_match("Saturn", "Jupiter"))                         # 0.0  ✅ correctly fails

## Step 6 — Metric #4: Token-level F1

`contains` is all-or-nothing. Often we want **partial credit**: 'Shakespeare'
got 1 of the 2 expected words right, so it deserves more than 0 but less than 1.

We borrow **precision**, **recall**, and **F1** from information retrieval,
applied to the *set of words (tokens)*:

- **Precision** = of the words the agent said, how many were expected? (penalizes rambling)
- **Recall** = of the expected words, how many did the agent say? (penalizes missing info)
- **F1** = the harmonic mean of the two — one balanced score.

This is the workhorse metric for short free-text answers in academic benchmarks
like SQuAD.

In [ ]:
def f1_score(prediction: str, expected: str) -> float:
    pred_tokens = normalize(prediction).split()
    exp_tokens  = normalize(expected).split()
    if not pred_tokens or not exp_tokens:
        return 0.0

    common = set(pred_tokens) & set(exp_tokens)
    if not common:
        return 0.0

    precision = len(common) / len(set(pred_tokens))
    recall    = len(common) / len(set(exp_tokens))
    return 2 * precision * recall / (precision + recall)

print(round(f1_score("Shakespeare", "William Shakespeare"), 2))          # 0.67  partial credit ✅
print(round(f1_score("The capital of France is Paris", "Paris"), 2))     # 0.29  right answer, lots of filler
print(round(f1_score("Jupiter", "Jupiter"), 2))                          # 1.0   perfect

## Step 7 — Metric #5: Latency & Cost

Quality is only half the story. A perfectly correct agent that takes 30 seconds
and costs $0.10 per call may be unusable. **Observability metrics** measure the
*operational* side: how long did it take, and how much did it cost?

We time the call with `time.perf_counter()` and estimate cost from token count.
(A rough rule of thumb: ~1 token ≈ 4 characters of English.)

In [ ]:
import time

def estimate_tokens(text: str) -> int:
    """Rough token estimate: ~4 characters per token for English."""
    return max(1, len(text) // 4)

def timed_call(fn, question: str):
    """Run fn(question), returning (answer, latency_seconds, est_tokens)."""
    start = time.perf_counter()
    answer = fn(question)
    latency = time.perf_counter() - start
    tokens = estimate_tokens(question) + estimate_tokens(answer)
    return answer, latency, tokens

# Example: an imaginary price of $0.05 per 1,000 tokens
PRICE_PER_1K_TOKENS = 0.05

answer, latency, tokens = timed_call(my_agent, "What is the largest planet?")
cost = tokens / 1000 * PRICE_PER_1K_TOKENS
print(f"answer  : {answer}")
print(f"latency : {latency*1000:.2f} ms")
print(f"tokens  : {tokens}  (est.)")
print(f"cost    : ${cost:.6f}  (est.)")

## Step 8 — The Eval Harness

Now we tie it all together. An **eval harness** is a loop that:
1. runs every test case through the system under test,
2. scores each output with every metric,
3. records latency & cost,
4. aggregates everything into a single **report**.

This is the reusable engine. Swap in a different `agent` or different `metrics`
and it just works — the same pattern scales from 4 cases to 4,000.

In [ ]:
def run_eval(agent, dataset, metrics):
    """Run `agent` over `dataset`, scoring with `metrics` (name -> function)."""
    rows = []
    for case in dataset:
        prediction, latency, tokens = timed_call(agent, case["input"])
        scores = {name: fn(prediction, case["expected"]) for name, fn in metrics.items()}
        rows.append({
            "id": case["id"],
            "input": case["input"],
            "expected": case["expected"],
            "prediction": prediction,
            "latency_ms": round(latency * 1000, 2),
            "tokens": tokens,
            "cost": round(tokens / 1000 * PRICE_PER_1K_TOKENS, 6),
            **scores,
        })

    # Aggregate: average each metric + totals for ops
    n = len(rows)
    summary = {name: round(sum(r[name] for r in rows) / n, 3) for name in metrics}
    summary["avg_latency_ms"] = round(sum(r["latency_ms"] for r in rows) / n, 2)
    summary["total_cost"] = round(sum(r["cost"] for r in rows), 6)
    return rows, summary

metrics = {
    "exact": exact_match,
    "normalized": normalized_match,
    "contains": contains_match,
    "f1": f1_score,
}

rows, summary = run_eval(my_agent, dataset, metrics)
print("Eval complete.")

## Step 9 — Read the report

Let's print the per-case results and the summary. Notice how the metrics tell
*different stories* about the same outputs — that's why you track several at
once. `exact` looks terrible; `contains` and `f1` reveal the agent is actually
mostly right (it just phrases things verbosely), except for the one genuinely
wrong planet answer.

In [ ]:
# Per-case table
header = f"{'id':<3} {'exact':<6} {'norm':<6} {'cont':<6} {'f1':<6} {'ms':<7} prediction"
print(header)
print('-' * len(header))
for r in rows:
    print(f"{r['id']:<3} {r['exact']:<6} {r['normalized']:<6} {r['contains']:<6} "
          f"{r['f1']:<6.2f} {r['latency_ms']:<7} {r['prediction'][:40]!r}")

print('\nSUMMARY (averaged over all cases)')
for k, v in summary.items():
    print(f"  {k:<16}: {v}")

### How to read these numbers

- **exact ≈ 0.25** — only the '144' answer matched character-for-character. By
  itself this would scare you, but it's misleading for free text.
- **contains / f1 much higher** — they correctly give credit for 'Paris' buried
  in a sentence and partial credit for 'Shakespeare'.
- **The planet question scores 0 everywhere** — that's a *real* bug the eval
  caught. This is exactly the kind of regression you want a number to flag.

> **Lesson:** no single metric is 'the truth'. Pick metrics that match your task,
> and read them together. For open-ended answers where even F1 falls short, you
> need a smarter grader — that's **LLM-as-a-judge** in notebook 02.

## Step 10 — Try it yourself

Three quick experiments to lock in the ideas:
1. **Fix the agent.** Change `"Saturn"` to `"Jupiter"` in `my_agent` and re-run
   Steps 8–9. Watch every metric climb.
2. **Add a case.** Append a new question to `dataset` and re-run.
3. **Add a metric.** Write a `length_ratio` metric (prediction length / expected
   length) and plug it into the `metrics` dict — the harness needs no other changes.

In [ ]:
# Your playground — edit and re-run.
def length_ratio(prediction: str, expected: str) -> float:
    """How many times longer is the answer than expected? (1.0 = same length)"""
    if not expected:
        return 0.0
    return round(len(prediction) / len(expected), 2)

my_metrics = {**metrics, "length_ratio": length_ratio}
rows2, summary2 = run_eval(my_agent, dataset, my_metrics)
print('New summary with your custom metric:')
for k, v in summary2.items():
    print(f"  {k:<16}: {v}")

## Summary

| What you built | What it taught |
|----------------|----------------|
| A `dataset` of test cases | Evaluation always starts from ground truth |
| `exact_match` | Strict string equality — great for labels/IDs, brutal for free text |
| `normalized_match` | Ignore case/punctuation noise before comparing |
| `contains_match` | Did the key fact appear at all? |
| `f1_score` | Partial credit via precision + recall over tokens |
| `timed_call` + cost estimate | Quality isn't enough — track latency & cost too |
| `run_eval` harness | One reusable loop: run → score → aggregate → report |

**The big takeaway:** an evaluation is just a comparison run over a dataset and
averaged. Everything else — LLM judges, RAG metrics, full pipelines — is a
fancier `compare()` function plugged into this exact same harness.

**Next up → `02-llm-as-judge`:** what do you do when the answer is an essay and
no string metric can tell good from bad? You let an LLM be the judge.